## BÁO CÁO Cuối KỲ - PHÂN TÍCH DỮ LIỆU VÀ AI

## Nhóm thực hiện: Nhóm 11

## MSSV: 24710631

## GV Hướng Dẫn: TS.Ngô Hữu Dũng

## Dataset: Road Traffic Accidents


## “Phân tích và dự đoán mức độ nghiêm trọng của tai nạn giao thông bằng Machine Learning.”

## 1. Tổng quan và chuẩn bị dữ liệu

In [ ]:

# 1.1 IMPORT THU VIEN

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.metrics import f1_score

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("Import thu vien thanh cong!")


In [ ]:

# 1.2. KIEM TRA DUONG DAN DATASET

import os

print(os.listdir('/kaggle/input'))


In [ ]:

# 1.3. DOC DU LIEU GOC

df = pd.read_csv(
    '/kaggle/input/datasets/saurabhshahane/road-traffic-accidents/RTA Dataset.csv'
)

print("Doc du lieu thanh cong!")


In [ ]:

# 1.4. HIEN THI 5 DONG DAU TIEN

df.head()


In [ ]:

# 1.5. KICH THUOC DU LIEU

print("So dong:", df.shape[0])
print("So cot:", df.shape[1])


In [ ]:

#1.6. THONG TIN DU LIEU

df.info()


## 2. Khám phá và phân tích dữ liệu (EDA)

In [ ]:

# 2.1.KIEM TRA GIA TRI THIEU

missing_data = pd.DataFrame({
    'So luong thieu': df.isnull().sum(),
    'Ty le thieu (%)': (df.isnull().mean() * 100).round(2)
})

missing_data[missing_data['So luong thieu'] > 0].sort_values(
    by='So luong thieu',
    ascending=False
)


In [ ]:

#2.2. KIEM TRA DONG TRUNG LAP

print("So dong trung lap:", df.duplicated().sum())


In [ ]:
#2.3. SAP XEP THU TU MUC DO NGHIEM TRONG

severity_order = [
    'Slight Injury',
    'Serious Injury',
    'Fatal injury'
]

## 3. Trực quan hóa dữ liệu

In [ ]:
#3.1. PHAN BO MUC DO NGHIEM TRONG TAI NAN

plt.figure(figsize=(8, 5))

ax = sns.countplot(
    data=df,
    x='Accident_severity',
    order=severity_order
)

for container in ax.containers:
    ax.bar_label(container)

plt.title('Phan bo muc do nghiem trong tai nan')
plt.xlabel('Muc do tai nan')
plt.ylabel('So luong')
plt.tight_layout()
plt.show()

In [ ]:
# 3.2. MUC DO TAI NAN THEO NGUYEN NHAN

# Lay 10 nguyen nhan co so vu tai nan nhieu nhat
top_causes = df['Cause_of_accident'].value_counts().head(10).index

cause_data = df[df['Cause_of_accident'].isin(top_causes)]

# Tinh ty le cac muc do tai nan theo tung nguyen nhan
cause_severity = pd.crosstab(
    cause_data['Cause_of_accident'],
    cause_data['Accident_severity'],
    normalize='index'
) * 100

# Sap xep cot va dong
cause_severity = cause_severity.reindex(
    index=top_causes,
    columns=severity_order,
    fill_value=0
)

# Lay so luong vu tai nan cua tung nguyen nhan
cause_counts = cause_data['Cause_of_accident'].value_counts()

# Them so luong vao ten nguyen nhan
cause_severity.index = [
    f'{cause} (n={cause_counts[cause]})'
    for cause in cause_severity.index
]

# Ve Heatmap
plt.figure(figsize=(12, 7))

sns.heatmap(
    cause_severity,
    annot=True,
    fmt='.1f',
    cmap='YlOrRd',
    linewidths=0.5
)

plt.title('Ty le muc do nghiem trong theo nguyen nhan tai nan')
plt.xlabel('Muc do tai nan')
plt.ylabel('Nguyen nhan tai nan')
plt.tight_layout()
plt.show()

In [ ]:
# 3.3. TAO COT GIO TU THOI GIAN TAI NAN

df['Hour'] = pd.to_datetime(
    df['Time'],
    format='%H:%M:%S',
    errors='coerce'
).dt.hour

In [ ]:
# 3.4. MUC DO TAI NAN THEO GIO

hour_severity = pd.crosstab(
    df['Hour'],
    df['Accident_severity'],
    normalize='index'
) * 100

hour_severity = hour_severity.reindex(
    columns=severity_order,
    fill_value=0
)

hour_severity.plot(
    kind='line',
    marker='o',
    figsize=(11, 6)
)

plt.title('Ty le muc do nghiem trong tai nan theo gio')
plt.xlabel('Gio trong ngay')
plt.ylabel('Ty le (%)')
plt.xticks(range(24))
plt.legend(title='Muc do tai nan')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# 3.5. MUC DO TAI NAN THEO DIEU KIEN ANH SANG

light_severity = pd.crosstab(
    df['Light_conditions'],
    df['Accident_severity'],
    normalize='index'
) * 100

light_severity = light_severity.reindex(
    columns=severity_order,
    fill_value=0
)

light_severity.plot(
    kind='bar',
    figsize=(11, 6),
    width=0.8
)

plt.title('Ty le muc do nghiem trong theo dieu kien anh sang')
plt.xlabel('Dieu kien anh sang')
plt.ylabel('Ty le (%)')
plt.xticks(rotation=25, ha='right')
plt.legend(title='Muc do tai nan')
plt.tight_layout()
plt.show()

In [ ]:
# 3.6. MUC DO TAI NAN THEO DIEU KIEN THOI TIET

weather_severity = pd.crosstab(
    df['Weather_conditions'],
    df['Accident_severity'],
    normalize='index'
) * 100

weather_severity = weather_severity.reindex(
    columns=severity_order,
    fill_value=0
)

weather_severity.plot(
    kind='bar',
    figsize=(12, 6),
    width=0.8
)

plt.title('Ty le muc do nghiem trong theo dieu kien thoi tiet')
plt.xlabel('Dieu kien thoi tiet')
plt.ylabel('Ty le (%)')
plt.xticks(rotation=25, ha='right')
plt.legend(title='Muc do tai nan')
plt.tight_layout()
plt.show()

## 4. Tiền xử lý dữ liệu

In [ ]:
# 4.1. KIEM TRA CAC GIA TRI TRONG CAC COT PHAN LOAI

categorical_cols = df.select_dtypes(include='object').columns

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(15))

In [ ]:
#4.2. LAM SACH DU LIEU DANG CHU

# Xoa khoang trang thua
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

# Chuan hoa cac gia tri Unknown
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace(
        to_replace=r'(?i)^unknown$',
        value='Unknown',
        regex=True
    )

# Thay gia tri thieu bang Unknown
df = df.fillna('Unknown')

print("Da lam sach du lieu!")
print("\nSo luong gia tri thieu sau khi lam sach:")
print(df.isnull().sum().sum())

print("\nKich thuoc du lieu:")
print(df.shape)

In [ ]:
#4.3. CHON CAC COT DAU VAO VA BIEN MUC TIEU

feature_cols = [
    'Hour',
    'Day_of_week',
    'Age_band_of_driver',
    'Sex_of_driver',
    'Educational_level',
    'Vehicle_driver_relation',
    'Driving_experience',
    'Type_of_vehicle',
    'Owner_of_vehicle',
    'Service_year_of_vehicle',
    'Defect_of_vehicle',
    'Area_accident_occured',
    'Lanes_or_Medians',
    'Road_allignment',
    'Types_of_Junction',
    'Road_surface_type',
    'Road_surface_conditions',
    'Light_conditions',
    'Weather_conditions',
    'Type_of_collision',
    'Vehicle_movement',
    'Pedestrian_movement',
    'Cause_of_accident'
]

# Tao X va y
X = df[feature_cols]
y = df['Accident_severity']

print("So dong du lieu:", X.shape[0])
print("So cot dau vao:", X.shape[1])

print("\nPhan bo bien muc tieu:")
print(y.value_counts())

print("\nTy le phan tram:")
print((y.value_counts(normalize=True) * 100).round(2))

In [ ]:
# 4.4. CHIA DU LIEU TRAIN VA TEST

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("So dong tap Train:", X_train.shape[0])
print("So dong tap Test:", X_test.shape[0])

print("\nTy le cac nhom trong tap Train:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTy le cac nhom trong tap Test:")
print((y_test.value_counts(normalize=True) * 100).round(2))

In [ ]:
# 4.5. XAC DINH CAC COT PHAN LOAI

categorical_cols = X_train.select_dtypes(
    include='object'
).columns.tolist()

# TAO BO MA HOA ONE-HOT
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)

# HUAN LUYEN BO MA HOA TREN TAP TRAIN
X_train_encoded = preprocessor.fit_transform(X_train)

# MA HOA TAP TEST
X_test_encoded = preprocessor.transform(X_test)

print("So cot truoc khi ma hoa:", X_train.shape[1])
print("So cot sau khi ma hoa:", X_train_encoded.shape[1])

print("Kich thuoc X_train:", X_train_encoded.shape)
print("Kich thuoc X_test:", X_test_encoded.shape)

## 5. Xây dựng và huấn luyện mô hình

In [ ]:
# 5.1 KHOI TAO CAC MO HINH

models = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ),

    'Decision Tree': DecisionTreeClassifier(
        class_weight='balanced',
        random_state=42
    ),

    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
}

# HUAN LUYEN CAC MO HINH
for name, model in models.items():
    model.fit(X_train_encoded, y_train)
    print(name, "- Huấn luyện thành công!")

In [ ]:
# CHUAN HOA DU LIEU CHO LOGISTIC REGRESSION

scaler = StandardScaler(with_mean=False)

X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

# HUAN LUYEN LAI LOGISTIC REGRESSION
models['Logistic Regression'] = LogisticRegression(
    class_weight='balanced',
    max_iter=3000,
    random_state=42
)

models['Logistic Regression'].fit(
    X_train_scaled,
    y_train
)

print("Logistic Regression - Huấn luyện lại thành công!")

## 6. Đánh giá mô hình

In [ ]:
#6.1. DU DOAN VA DANH GIA CAC MO HINH

results = []

for name, model in models.items():

    # Chon du lieu phu hop voi tung mo hinh
    if name == 'Logistic Regression':
        X_test_model = X_test_scaled
    else:
        X_test_model = X_test_encoded

    # Du doan
    y_pred = model.predict(X_test_model)

    # Tinh cac chi so danh gia
    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    weighted_f1 = f1_score(y_test, y_pred, average='weighted')

    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Macro F1': macro_f1,
        'Weighted F1': weighted_f1
    })

# Tao bang ket qua
results_df = pd.DataFrame(results)

print("BANG SO SANH CAC MO HINH:")
display(results_df.round(4))

In [ ]:
# 6.2. BAO CAO CHI TIET CAC MO HINH

for name, model in models.items():

    if name == 'Logistic Regression':
        X_test_model = X_test_scaled
    else:
        X_test_model = X_test_encoded

    y_pred = model.predict(X_test_model)

    print("\nMO HINH:", name)
    print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# 6.3. VE MA TRAN NHAM LAN CUA 3 MO HINH

for name, model in models.items():

    if name == 'Logistic Regression':
        X_test_model = X_test_scaled
    else:
        X_test_model = X_test_encoded

    y_pred = model.predict(X_test_model)

    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=['Fatal injury', 'Serious Injury', 'Slight Injury']
    )

    plt.figure(figsize=(6, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['Fatal', 'Serious', 'Slight'],
        yticklabels=['Fatal', 'Serious', 'Slight']
    )

    plt.title(f'Confusion Matrix - {name}')
    plt.xlabel('Gia tri du doan')
    plt.ylabel('Gia tri thuc te')
    plt.tight_layout()
    plt.show()

## 7. Tối ưu mô hình

In [ ]:
# 7.1. Tối ưu Decision Tree bằng GridSearchCV và Cross-Validation
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 10],
    'min_samples_leaf': [1, 5, 10]
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(
        class_weight='balanced',
        random_state=42
    ),
    param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1
)

grid_search.fit(X_train_encoded, y_train)

best_model = grid_search.best_estimator_

print("Tham số tốt nhất:", grid_search.best_params_)
print("Macro F1 tốt nhất:", grid_search.best_score_)

In [ ]:
# 7.2. Đánh giá Decision Tree sau tối ưu
from sklearn.metrics import accuracy_score, f1_score, classification_report

y_pred_best = best_model.predict(X_test_encoded)

print("Accuracy:", accuracy_score(y_test, y_pred_best))
print("Macro F1:", f1_score(y_test, y_pred_best, average='macro'))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best))

## 8. Tổng kết kết quả

# 8.1. Tổng hợp kết quả thực nghiệm

Trong quá trình thực nghiệm, nhóm đã sử dụng ba thuật toán Machine Learning để xây dựng mô hình dự đoán mức độ nghiêm trọng của tai nạn giao thông, bao gồm Logistic Regression, Decision Tree và Random Forest.

Kết quả đánh giá trên tập kiểm tra gồm 2.464 mẫu:

| Mô hình | Accuracy | Macro F1 |
|---|---:|---:|
| Logistic Regression | 46% | 0,30 |
| Decision Tree | 72% | **0,36** |
| Random Forest | **85%** | 0,31 |

Qua kết quả trên, có thể nhận thấy:

- **Logistic Regression:** Đạt Accuracy 46%, Macro F1 0,30. Mô hình có khả năng nhận diện một số trường hợp tai nạn nghiêm trọng nhưng kết quả dự đoán tổng thể còn hạn chế.
- **Decision Tree:** Đạt Accuracy khoảng 72%, Macro F1 khoảng 0,36. Đây là mô hình có Macro F1 cao nhất trong ba thuật toán được thử nghiệm.
- **Random Forest:** Đạt Accuracy khoảng 85%, nhưng Macro F1 chỉ đạt 0,31. Mô hình chủ yếu dự đoán các trường hợp Slight Injury và không nhận diện được hai nhóm Serious Injury và Fatal Injury trong tập kiểm tra.

Nhóm tiếp tục sử dụng GridSearchCV kết hợp Cross-Validation để tối ưu Decision Tree. Kết quả lựa chọn bộ tham số `max_depth=None`, `min_samples_split=2`, `min_samples_leaf=1`, với Macro F1 trung bình đạt 0,3471.

Sau tối ưu, mô hình đạt Accuracy 72,40% và Macro F1 0,3576 trên tập kiểm tra. Kết quả gần như không thay đổi so với mô hình Decision Tree ban đầu.

# 8.2. Hạn chế của mô hình

Mặc dù đã xây dựng và đánh giá được ba mô hình dự đoán, đề tài vẫn còn một số hạn chế:

- **Dữ liệu mất cân bằng:** Số lượng mẫu Slight Injury chiếm khoảng 84,56%, trong khi Fatal Injury chỉ chiếm khoảng 1,28%. Điều này khiến mô hình khó học được đặc điểm của nhóm tai nạn nghiêm trọng.
- **Khả năng nhận diện nhóm thiểu số còn thấp:** Các mô hình chưa dự đoán tốt hai nhóm Serious Injury và Fatal Injury, đặc biệt là Random Forest.
- **Kết quả tối ưu chưa cải thiện đáng kể:** Việc sử dụng GridSearchCV chưa giúp Decision Tree nâng cao rõ rệt hiệu quả dự đoán.
- **Phạm vi dữ liệu còn giới hạn:** Bộ dữ liệu được thu thập từ các vụ tai nạn tại thành phố Addis Ababa, Ethiopia trong giai đoạn 2017–2020. Vì vậy, kết quả chưa thể đại diện cho tình hình tai nạn giao thông ở các khu vực khác.

# 8.3. Kết luận và hướng phát triển

Đề tài đã thực hiện quy trình phân tích dữ liệu và ứng dụng Machine Learning để dự đoán mức độ nghiêm trọng của tai nạn giao thông. Nhóm đã tiến hành khám phá dữ liệu, trực quan hóa, xử lý dữ liệu thiếu, mã hóa các thuộc tính phân loại, chia tập dữ liệu và xây dựng ba mô hình Logistic Regression, Decision Tree và Random Forest.

Thông qua quá trình đánh giá, Decision Tree được lựa chọn làm mô hình phù hợp nhất trong ba thuật toán đã thử nghiệm vì đạt Macro F1 cao nhất, khoảng 0,36. Tuy nhiên, hiệu quả nhận diện các trường hợp tai nạn nghiêm trọng và tử vong vẫn còn thấp do sự mất cân bằng giữa các lớp dữ liệu.

Trong tương lai, đề tài có thể được phát triển bằng cách bổ sung dữ liệu tai nạn từ nhiều khu vực và thời điểm khác nhau, thử nghiệm các phương pháp xử lý dữ liệu mất cân bằng, đồng thời nghiên cứu thêm các thuật toán và phương pháp tối ưu nhằm cải thiện khả năng nhận diện những vụ tai nạn có mức độ nghiêm trọng cao.